- Interactive Python Notebook. 
- The notebook must be functional and runnable (i.e., we must be able to import the .ipynyb without your support, and execute it).
- The results produced by the notebook should be self-explanatory.

# Compression codecs comparison analysis

Group members
- Laila Ibrahim
- Verica Dimitrova

# Importing the relevant elements for our analysis

In [159]:
# The libraries 

## Make sure they're already installed on the machine. 
##Otherwise we'll get an error that the library/module does not exist (no such thing as...)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [160]:
# The data

## results.csv (obtained when benchmarking MinIo)
minio = pd.read_csv("results.csv")
minio["codec"] = minio["codec"].fillna("None")

## results_with_Azure.csv
azure = pd.read_csv("results_with_Azure.csv")
azure["codec"] = azure["codec"].fillna("None")

# Data Manipulation

## MinIO

In [161]:
# Convert the stored bytes into Megabytes (more intuitive)
# 1 Mb -> 1_048_576 bytes 
# y Mb -> x bytes
# y = (x*1)/1_048_576

# Created a new column for that
minio["stored_Mb"] = (minio["stored_bytes"])/1_048_576

# Reorder the columns to put 'stored_Mb' next to 'stored_bytes'
minio = minio[['size'] + ['codec', 'stored_bytes', 'stored_Mb'] 
              + [col for col in minio.columns if col not in ['size','codec', 'stored_bytes', 'stored_Mb']]]

In [162]:
# Separating the results according to the size of the parquets
minio_S = minio[minio["size"] == "S"] 
minio_M = minio[minio["size"] == "M"]
minio_L = minio[minio["size"] == "L"]

# Azure

We will go through the same process as with MinIo

In [163]:
# Convert the stored bytes into Megabytes and save them in a new column
azure["stored_Mb"] = (azure["stored_bytes"])/1_048_576

# Reorder the columns to put 'stored_Mb' next to 'stored_bytes'
azure = azure[['size'] + ['codec', 'stored_bytes', 'stored_Mb'] 
              + [col for col in azure.columns if col not in ['size','codec', 'stored_bytes', 'stored_Mb']]]

In [164]:
# Separating the results according to the size of the parquets
azure_S = azure[azure["size"] == "S"] 
azure_M = azure[azure["size"] == "M"]
azure_L = azure[azure["size"] == "L"]

# Compression ratio

- We'll compute the compression ratio for both MinIo and Azure.
- Useful to make sure if it's actually working or not
- *Formula*: 
compression ratio = uncompressed size/ compressed size

## MinIo

In [165]:
# The .loc[] is used to access specific rows and columns in a pandas DataFrame. (.iloc[] also does that)

uncompressed_S_minio = minio.loc[(minio['size'] == 'S') & (minio['codec'] == 'None'), 'stored_Mb'].values[0]
uncompressed_M_minio = minio.loc[(minio['size'] == 'M') & (minio['codec'] == 'None'), 'stored_Mb'].values[0]
uncompressed_L_minio = minio.loc[(minio['size'] == 'L') & (minio['codec'] == 'None'), 'stored_Mb'].values[0]

compression_ratios_S_minio = {
    "snappy": uncompressed_S_minio / minio.loc[(minio['size'] == 'S') & (minio['codec'] == 'snappy'), 'stored_Mb'].values[0],
    "zstd":   uncompressed_S_minio / minio.loc[(minio['size'] == 'S') & (minio['codec'] == 'zstd'), 'stored_Mb'].values[0],
    "gzip":   uncompressed_S_minio / minio.loc[(minio['size'] == 'S') & (minio['codec'] == 'gzip'), 'stored_Mb'].values[0],
}

compression_ratios_M_minio = {
    "snappy": uncompressed_M_minio / minio.loc[(minio['size'] == 'M') & (minio['codec'] == 'snappy'), 'stored_Mb'].values[0],
    "zstd":   uncompressed_M_minio / minio.loc[(minio['size'] == 'M') & (minio['codec'] == 'zstd'), 'stored_Mb'].values[0],
    "gzip":   uncompressed_M_minio / minio.loc[(minio['size'] == 'M') & (minio['codec'] == 'gzip'), 'stored_Mb'].values[0],
}

compression_ratios_L_minio = {
    "snappy": uncompressed_L_minio / minio.loc[(minio['size'] == 'L') & (minio['codec'] == 'snappy'), 'stored_Mb'].values[0],
    "zstd":   uncompressed_L_minio / minio.loc[(minio['size'] == 'L') & (minio['codec'] == 'zstd'), 'stored_Mb'].values[0],
    "gzip":   uncompressed_L_minio / minio.loc[(minio['size'] == 'L') & (minio['codec'] == 'gzip'), 'stored_Mb'].values[0],
}

In [166]:
# Plotting the ratios to be able to see how mush is it for each codec and each size
codecs = ["snappy", "zstd", "gzip"]
x = np.arange(len(codecs)) * 2
width = 0.5

fig, ax = plt.subplots(figsize=(12, 6))

bars_S_minio = ax.bar(x - width, [compression_ratios_S_minio[c] for c in codecs], width, label="S")
bars_M_minio = ax.bar(x,         [compression_ratios_M_minio[c] for c in codecs], width, label="M")
bars_L_minio = ax.bar(x + width, [compression_ratios_L_minio[c] for c in codecs], width, label="L")

ax.set_xticks(x)
ax.set_xticklabels(codecs)
ax.set_ylabel("Compression Ratio")
ax.set_title("Compression Ratio per Codec (S, M & L)")
ax.legend()
ax.bar_label(bars_S_minio, fmt="%.3f", padding=3)
ax.bar_label(bars_M_minio, fmt="%.3f", padding=3)
ax.bar_label(bars_L_minio, fmt="%.3f", padding=3)

plt.tight_layout()
plt.show()

## Azure

In [167]:
uncompressed_S_azure = azure.loc[(azure['size'] == 'S') & (azure['codec'] == 'None'), 'stored_Mb'].values[0]
uncompressed_M_azure = azure.loc[(azure['size'] == 'M') & (azure['codec'] == 'None'), 'stored_Mb'].values[0]
uncompressed_L_azure = azure.loc[(azure['size'] == 'L') & (azure['codec'] == 'None'), 'stored_Mb'].values[0]

compression_ratios_S_azure = {
    "snappy": uncompressed_S_azure / azure.loc[(azure['size'] == 'S') & (azure['codec'] == 'snappy'), 'stored_Mb'].values[0],
    "zstd":   uncompressed_S_azure / azure.loc[(azure['size'] == 'S') & (azure['codec'] == 'zstd'), 'stored_Mb'].values[0],
    "gzip":   uncompressed_S_azure / azure.loc[(azure['size'] == 'S') & (azure['codec'] == 'gzip'), 'stored_Mb'].values[0],
}

compression_ratios_M_azure = {
    "snappy": uncompressed_M_azure / azure.loc[(azure['size'] == 'M') & (azure['codec'] == 'snappy'), 'stored_Mb'].values[0],
    "zstd":   uncompressed_M_azure / azure.loc[(azure['size'] == 'M') & (azure['codec'] == 'zstd'), 'stored_Mb'].values[0],
    "gzip":   uncompressed_M_azure / azure.loc[(azure['size'] == 'M') & (azure['codec'] == 'gzip'), 'stored_Mb'].values[0],
}

compression_ratios_L_azure = {
    "snappy": uncompressed_L_azure / azure.loc[(azure['size'] == 'L') & (azure['codec'] == 'snappy'), 'stored_Mb'].values[0],
    "zstd":   uncompressed_L_azure / azure.loc[(azure['size'] == 'L') & (azure['codec'] == 'zstd'), 'stored_Mb'].values[0],
    "gzip":   uncompressed_L_azure / azure.loc[(azure['size'] == 'L') & (azure['codec'] == 'gzip'), 'stored_Mb'].values[0],
}


In [168]:
codecs = ["snappy", "zstd", "gzip"]
x = np.arange(len(codecs)) * 2
width = 0.5

fig, ax = plt.subplots(figsize=(12, 6))

bars_S_azure = ax.bar(x - width, [compression_ratios_S_azure[c] for c in codecs], width, label="S")
bars_M_azure = ax.bar(x,         [compression_ratios_M_azure[c] for c in codecs], width, label="M")
bars_L_azure = ax.bar(x + width, [compression_ratios_L_azure[c] for c in codecs], width, label="L")

ax.set_xticks(x)
ax.set_xticklabels(codecs)
ax.set_ylabel("Compression Ratio")
ax.set_title("Compression Ratio per Codec (S, M & L)")
ax.legend()
ax.bar_label(bars_S_azure, fmt="%.3f", padding=3)
ax.bar_label(bars_M_azure, fmt="%.3f", padding=3)
ax.bar_label(bars_L_azure, fmt="%.3f", padding=3)

plt.tight_layout()
plt.show()

# Interactive Plots

In [169]:
# Putting those here because we'll use for all the interactive plots (MinIo AND Azure)
# By putting them here, we execute it once and c'est bon.
# This cell HAS TO BE EXECUTED to be able to get all the dynamic plots that follow
codecs = minio_M['codec'] # doesn't matter if it's minio_M, minio_S or minio_L (or azure_S, azure_M or azure_L) since the codecs are the same at the end
sizes = ['S', 'M', 'L']

## MinIo

### Total stored bytes per codec & per size

In [170]:
#  Stored bytes

stored_bytes_minio = {
    'S': minio_S["stored_bytes"],
    'M': minio_M["stored_bytes"],
    'L': minio_L["stored_bytes"]
}

fig = go.Figure()

# 1. Add all traces initially (we will control visibility via dropdowns)
for size in sizes:
    fig.add_trace(
        go.Bar(
            x=codecs,
            y=stored_bytes_minio[size],
            name=f"Size {size}",
            text=[f"{v:.3f}" for v in stored_bytes_minio[size]],
            textposition='auto'
        )
    )

# 2. Define the Dropdown Menus
fig.update_layout(
    updatemenus=[
        # Dropdown 1: Select File Size (S, M, L, or All)
        dict(
            buttons=[
                dict(label="All Sizes", method="update", args=[{"visible": [True, True, True]}, {"title": "Stored bytes: All Sizes (Dataset: MinIo)"}]),
                dict(label="Size S", method="update", args=[{"visible": [True, False, False]}, {"title": "Stored bytes: Size S"}]),
                dict(label="Size M", method="update", args=[{"visible": [False, True, False]}, {"title": "Stored bytes: Size M"}]),
                dict(label="Size L", method="update", args=[{"visible": [False, False, True]}, {"title": "Stored bytes: Size L"}]),
            ],
            direction="down",
            showactive=True,
            x=0.1, xanchor="left", y=1.15, yanchor="top"
        ),
        # Dropdown 2: Codec Filter (Restricts view to specific codec AND updates title)
        dict(
            buttons=[
                dict(
                    label="All Codecs", 
                    method="update", 
                    args=[
                        {}, # No changes to trace data
                        {"xaxis.range": [-0.5, len(codecs)-0.5], "title": "Stored bytes: All Codecs (Dataset: MinIo)"}
                    ]
                ),
                *[
                    dict(
                        label=f"Codec: {c}",
                        method="update",
                        args=[
                            {}, # No changes to trace data
                            {
                                "xaxis.range": [i-0.5, i+0.5], 
                                "title": f"Stored bytes: Codec {c}"
                            }
                        ]
                    ) for i, c in enumerate(codecs)
                ]
            ],
            direction="down",
            showactive=True,
            x=0.4, xanchor="left", y=1.15, yanchor="top"
        )
    ],
    title_text="Stored bytes per Codec and Size (Dataset: MinIo)",
    xaxis_title="Codec",
    yaxis_title="bytes",
    barmode='group', # Keep them grouped when "All Sizes" is selected
    template="plotly_white"
)

fig.show()

### Total stored Mb per codec & per size

In [171]:
#  Stored Mb (more intuitive)

stored_Mb_minio = {
    'S': minio_S["stored_Mb"],
    'M': minio_M["stored_Mb"],
    'L': minio_L["stored_Mb"]
}

fig = go.Figure()

# 1. Add all traces initially (we will control visibility via dropdowns)
for size in sizes:
    fig.add_trace(
        go.Bar(
            x=codecs,
            y=stored_Mb_minio[size],
            name=f"Size {size}",
            text=[f"{v:.3f}" for v in stored_Mb_minio[size]],
            textposition='auto'
        )
    )

# 2. Define the Dropdown Menus
fig.update_layout(
    updatemenus=[
        # Dropdown 1: Select File Size (S, M, L, or All)
        dict(
            buttons=[
                dict(label="All Sizes", method="update", args=[{"visible": [True, True, True]}, {"title": "Stored Mb: All Sizes (Dataset: MinIo)"}]),
                dict(label="Size S", method="update", args=[{"visible": [True, False, False]}, {"title": "Stored Mb: Size S"}]),
                dict(label="Size M", method="update", args=[{"visible": [False, True, False]}, {"title": "Stored Mb: Size M"}]),
                dict(label="Size L", method="update", args=[{"visible": [False, False, True]}, {"title": "Stored Mb: Size L"}]),
            ],
            direction="down",
            showactive=True,
            x=0.1, xanchor="left", y=1.15, yanchor="top"
        ),
        # Dropdown 2: Codec Filter (Restricts view to specific codec AND updates title)
        dict(
            buttons=[
                dict(
                    label="All Codecs", 
                    method="update", 
                    args=[
                        {}, # No changes to trace data
                        {"xaxis.range": [-0.5, len(codecs)-0.5], "title": "Stored Mb: All Codecs (Dataset: MinIo)"}
                    ]
                ),
                *[
                    dict(
                        label=f"Codec: {c}",
                        method="update",
                        args=[
                            {}, # No changes to trace data
                            {
                                "xaxis.range": [i-0.5, i+0.5], 
                                "title": f"Stored Mb: Codec {c}"
                            }
                        ]
                    ) for i, c in enumerate(codecs)
                ]
            ],
            direction="down",
            showactive=True,
            x=0.4, xanchor="left", y=1.15, yanchor="top"
        )
    ],
    title_text="Stored Mb per Codec and Size (Dataset: MinIo)",
    xaxis_title="Codec",
    yaxis_title="Mb",
    barmode='group', # Keep them grouped when "All Sizes" is selected
    template="plotly_white"
)

fig.show()

### Upload throughput per codec & per size

In [172]:
upload_throughput_minio = {
    'S': minio_S["upload_throughput"],
    'M': minio_M["upload_throughput"],
    'L': minio_L["upload_throughput"]
}

fig = go.Figure()

# 1. Add all traces initially (we will control visibility via dropdowns)
for size in sizes:
    fig.add_trace(
        go.Bar(
            x=codecs,
            y=upload_throughput_minio[size],
            name=f"Size {size}",
            text=[f"{v:.3f}" for v in upload_throughput_minio[size]],
            textposition='auto'
        )
    )

# 2. Define the Dropdown Menus
fig.update_layout(
    updatemenus=[
        # Dropdown 1: Select File Size (S, M, L, or All)
        dict(
            buttons=[
                dict(label="All Sizes", method="update", args=[{"visible": [True, True, True]}, {"title": "Upload Throughput: All Sizes (Dataset: MinIo)"}]),
                dict(label="Size S", method="update", args=[{"visible": [True, False, False]}, {"title": "Upload Throughput: Size S"}]),
                dict(label="Size M", method="update", args=[{"visible": [False, True, False]}, {"title": "Upload Throughput: Size M"}]),
                dict(label="Size L", method="update", args=[{"visible": [False, False, True]}, {"title": "Upload Throughput: Size L"}]),
            ],
            direction="down",
            showactive=True,
            x=0.1, xanchor="left", y=1.15, yanchor="top"
        ),
        # Dropdown 2: Codec Filter (Restricts view to specific codec)
        dict(
            buttons=[
                dict(
                    label="All Codecs", 
                    method="update", 
                    args=[
                        {}, # No changes to trace data
                        {"xaxis.range": [-0.5, len(codecs)-0.5], "title": "Upload Throughput: All Codecs (Dataset: MinIo)"}
                    ]
                ),
                *[
                    dict(
                        label=f"Codec: {c}",
                        method="update",
                        args=[
                            {}, # No changes to trace data
                            {
                                "xaxis.range": [i-0.5, i+0.5], 
                                "title": f"Query time: Codec {c}"
                            }
                        ]
                    ) for i, c in enumerate(codecs)
                ]
            ],
            direction="down",
            showactive=True,
            x=0.4, xanchor="left", y=1.15, yanchor="top"
        )
    ],
    title_text="Upload Throughput per Codec and Size (Dataset: MinIo)",
    xaxis_title="Codec",
    yaxis_title="Throughput (Mb/s)",
    barmode='group', # Keep them grouped when "All Sizes" is selected
    template="plotly_white"
)

fig.show()

### Download throughput per codec & per size

In [173]:
download_throughput_minio = {
    'S': minio_S["download_throughput"],
    'M': minio_M["download_throughput"],
    'L': minio_L["download_throughput"]
}

fig = go.Figure()

# 1. Add all traces initially (we will control visibility via dropdowns)
for size in sizes:
    fig.add_trace(
        go.Bar(
            x=codecs,
            y=download_throughput_minio[size],
            name=f"Size {size}",
            text=[f"{v:.3f}" for v in download_throughput_minio[size]],
            textposition='auto'
        )
    )

# 2. Define the Dropdown Menus
fig.update_layout(
    updatemenus=[
        # Dropdown 1: Select File Size (S, M, L, or All)
        dict(
            buttons=[
                dict(label="All Sizes", method="update", args=[{"visible": [True, True, True]}, {"title": "Download Throughput: All Sizes (Dataset: MinIo)"}]),
                dict(label="Size S", method="update", args=[{"visible": [True, False, False]}, {"title": "Download Throughput: Size S"}]),
                dict(label="Size M", method="update", args=[{"visible": [False, True, False]}, {"title": "Download Throughput: Size M"}]),
                dict(label="Size L", method="update", args=[{"visible": [False, False, True]}, {"title": "Download Throughput: Size L"}]),
            ],
            direction="down",
            showactive=True,
            x=0.1, xanchor="left", y=1.15, yanchor="top"
        ),
        # Dropdown 2: Codec Filter (Restricts view to specific codec)
        dict(
            buttons=[
                dict(
                    label="All Codecs", 
                    method="update", 
                    args=[
                        {}, # No changes to trace data
                        {"xaxis.range": [-0.5, len(codecs)-0.5], "title": "Download Throughput: All Codecs (Dataset: MinIo)"}
                    ]
                ),
                *[
                    dict(
                        label=f"Codec: {c}",
                        method="update",
                        args=[
                            {}, # No changes to trace data
                            {
                                "xaxis.range": [i-0.5, i+0.5], 
                                "title": f"Query time: Codec {c}"
                            }
                        ]
                    ) for i, c in enumerate(codecs)
                ]
            ],
            direction="down",
            showactive=True,
            x=0.4, xanchor="left", y=1.15, yanchor="top"
        )
    ],
    title_text="Download Throughput per Codec and Size (Dataset: MinIo)",
    xaxis_title="Codec",
    yaxis_title="Throughput (Mb/s)",
    barmode='group', # Keep them grouped when "All Sizes" is selected
    template="plotly_white"
)

fig.show()

### Query runtime per codec & per size

In [174]:
query_time_minio = {
    'S': minio_S["query_time_seconds"],
    'M': minio_M["query_time_seconds"],
    'L': minio_L["query_time_seconds"]
}

fig = go.Figure()

# 1. Add all traces initially (we will control visibility via dropdowns)
for size in sizes:
    fig.add_trace(
        go.Bar(
            x=codecs,
            y=query_time_minio[size],
            name=f"Size {size}",
            text=[f"{v:.3f}" for v in query_time_minio[size]],
            textposition='auto'
        )
    )

# 2. Define the Dropdown Menus
fig.update_layout(
    updatemenus=[
        # Dropdown 1: Select File Size (S, M, L, or All)
        dict(
            buttons=[
                dict(label="All Sizes", method="update", args=[{"visible": [True, True, True]}, {"title": "Query Time: All Sizes (Dataset: MinIo)"}]),
                dict(label="Size S", method="update", args=[{"visible": [True, False, False]}, {"title": "Query Time: Size S"}]),
                dict(label="Size M", method="update", args=[{"visible": [False, True, False]}, {"title": "Query Time: Size M"}]),
                dict(label="Size L", method="update", args=[{"visible": [False, False, True]}, {"title": "Query Time: Size L"}]),
            ],
            direction="down",
            showactive=True,
            x=0.1, xanchor="left", y=1.15, yanchor="top"
        ),
        # Dropdown 2: Codec Filter (Restricts view to specific codec)
        dict(
            buttons=[
                dict(
                    label="All Codecs", 
                    method="update", 
                    args=[
                        {}, # No changes to trace data
                        {"xaxis.range": [-0.5, len(codecs)-0.5], "title": "Query time: All Codecs (Dataset: MinIo)"}
                    ]
                ),
                *[
                    dict(
                        label=f"Codec: {c}",
                        method="update",
                        args=[
                            {}, # No changes to trace data
                            {
                                "xaxis.range": [i-0.5, i+0.5], 
                                "title": f"Query time: Codec {c}"
                            }
                        ]
                    ) for i, c in enumerate(codecs)
                ]
            ],
            direction="down",
            showactive=True,
            x=0.4, xanchor="left", y=1.15, yanchor="top"
        )
    ],
    title_text="Query time per Codec and Size (Dataset: MinIo)",
    xaxis_title="Codec",
    yaxis_title="Time (seconds)",
    barmode='group', # Keep them grouped when "All Sizes" is selected
    template="plotly_white"
)

fig.show()

## Azure

### Total stored bytes per codec & per size

In [175]:
#  Stored bytes

stored_bytes_azure = {
    'S': azure_S["stored_bytes"],
    'M': azure_M["stored_bytes"],
    'L': azure_L["stored_bytes"]
}

fig = go.Figure()

# 1. Add all traces initially (we will control visibility via dropdowns)
for size in sizes:
    fig.add_trace(
        go.Bar(
            x=codecs,
            y=stored_bytes_azure[size],
            name=f"Size {size}",
            text=[f"{v:.3f}" for v in stored_bytes_azure[size]],
            textposition='auto'
        )
    )

# 2. Define the Dropdown Menus
fig.update_layout(
    updatemenus=[
        # Dropdown 1: Select File Size (S, M, L, or All)
        dict(
            buttons=[
                dict(label="All Sizes", method="update", args=[{"visible": [True, True, True]}, {"title": "Stored bytes: All Sizes (Dataset: Azure)"}]),
                dict(label="Size S", method="update", args=[{"visible": [True, False, False]}, {"title": "Stored bytes: Size S"}]),
                dict(label="Size M", method="update", args=[{"visible": [False, True, False]}, {"title": "Stored bytes: Size M"}]),
                dict(label="Size L", method="update", args=[{"visible": [False, False, True]}, {"title": "Stored bytes: Size L"}]),
            ],
            direction="down",
            showactive=True,
            x=0.1, xanchor="left", y=1.15, yanchor="top"
        ),
        # Dropdown 2: Codec Filter (Restricts view to specific codec AND updates title)
        dict(
            buttons=[
                dict(
                    label="All Codecs", 
                    method="update", 
                    args=[
                        {}, # No changes to trace data
                        {"xaxis.range": [-0.5, len(codecs)-0.5], "title": "Stored bytes: All Codecs (Dataset: Azure)"}
                    ]
                ),
                *[
                    dict(
                        label=f"Codec: {c}",
                        method="update",
                        args=[
                            {}, # No changes to trace data
                            {
                                "xaxis.range": [i-0.5, i+0.5], 
                                "title": f"Stored bytes: Codec {c}"
                            }
                        ]
                    ) for i, c in enumerate(codecs)
                ]
            ],
            direction="down",
            showactive=True,
            x=0.4, xanchor="left", y=1.15, yanchor="top"
        )
    ],
    title_text="Stored bytes per Codec and Size (Dataset: Azure)",
    xaxis_title="Codec",
    yaxis_title="bytes",
    barmode='group', # Keep them grouped when "All Sizes" is selected
    template="plotly_white"
)

fig.show()

### Total stored Mb per codec & per size

In [176]:
#  Stored Mb (more intuitive)

stored_Mb_azure = {
    'S': azure_S["stored_Mb"],
    'M': azure_M["stored_Mb"],
    'L': azure_L["stored_Mb"]
}

fig = go.Figure()

# 1. Add all traces initially (we will control visibility via dropdowns)
for size in sizes:
    fig.add_trace(
        go.Bar(
            x=codecs,
            y=stored_Mb_azure[size],
            name=f"Size {size}",
            text=[f"{v:.3f}" for v in stored_Mb_azure[size]],
            textposition='auto'
        )
    )

# 2. Define the Dropdown Menus
fig.update_layout(
    updatemenus=[
        # Dropdown 1: Select File Size (S, M, L, or All)
        dict(
            buttons=[
                dict(label="All Sizes", method="update", args=[{"visible": [True, True, True]}, {"title": "Stored Mb: All Sizes (Dataset: Azure)"}]),
                dict(label="Size S", method="update", args=[{"visible": [True, False, False]}, {"title": "Stored Mb: Size S"}]),
                dict(label="Size M", method="update", args=[{"visible": [False, True, False]}, {"title": "Stored Mb: Size M"}]),
                dict(label="Size L", method="update", args=[{"visible": [False, False, True]}, {"title": "Stored Mb: Size L"}]),
            ],
            direction="down",
            showactive=True,
            x=0.1, xanchor="left", y=1.15, yanchor="top"
        ),
        # Dropdown 2: Codec Filter (Restricts view to specific codec AND updates title)
        dict(
            buttons=[
                dict(
                    label="All Codecs", 
                    method="update", 
                    args=[
                        {}, # No changes to trace data
                        {"xaxis.range": [-0.5, len(codecs)-0.5], "title": "Stored Mb: All Codecs (Dataset: Azure)"}
                    ]
                ),
                *[
                    dict(
                        label=f"Codec: {c}",
                        method="update",
                        args=[
                            {}, # No changes to trace data
                            {
                                "xaxis.range": [i-0.5, i+0.5], 
                                "title": f"Stored Mb: Codec {c}"
                            }
                        ]
                    ) for i, c in enumerate(codecs)
                ]
            ],
            direction="down",
            showactive=True,
            x=0.4, xanchor="left", y=1.15, yanchor="top"
        )
    ],
    title_text="Stored Mb per Codec and Size (Dataset: Azure)",
    xaxis_title="Codec",
    yaxis_title="Mb",
    barmode='group', # Keep them grouped when "All Sizes" is selected
    template="plotly_white"
)

fig.show()

### Upload throughput per codec & per size

In [177]:
upload_throughput_azure = {
    'S': azure_S["upload_throughput"],
    'M': azure_M["upload_throughput"],
    'L': azure_L["upload_throughput"]
}

fig = go.Figure()

# 1. Add all traces initially (we will control visibility via dropdowns)
for size in sizes:
    fig.add_trace(
        go.Bar(
            x=codecs,
            y=upload_throughput_azure[size],
            name=f"Size {size}",
            text=[f"{v:.3f}" for v in upload_throughput_azure[size]],
            textposition='auto'
        )
    )

# 2. Define the Dropdown Menus
fig.update_layout(
    updatemenus=[
        # Dropdown 1: Select File Size (S, M, L, or All)
        dict(
            buttons=[
                dict(label="All Sizes", method="update", args=[{"visible": [True, True, True]}, {"title": "Upload Throughput: All Sizes (Dataset: Azure)"}]),
                dict(label="Size S", method="update", args=[{"visible": [True, False, False]}, {"title": "Upload Throughput: Size S"}]),
                dict(label="Size M", method="update", args=[{"visible": [False, True, False]}, {"title": "Upload Throughput: Size M"}]),
                dict(label="Size L", method="update", args=[{"visible": [False, False, True]}, {"title": "Upload Throughput: Size L"}]),
            ],
            direction="down",
            showactive=True,
            x=0.1, xanchor="left", y=1.15, yanchor="top"
        ),
        # Dropdown 2: Codec Filter (Restricts view to specific codec)
        dict(
            buttons=[
                dict(
                    label="All Codecs", 
                    method="update", 
                    args=[
                        {}, # No changes to trace data
                        {"xaxis.range": [-0.5, len(codecs)-0.5], "title": "Upload Throughput: All Codecs (Dataset: Azure)"}
                    ]
                ),
                *[
                    dict(
                        label=f"Codec: {c}",
                        method="update",
                        args=[
                            {}, # No changes to trace data
                            {
                                "xaxis.range": [i-0.5, i+0.5], 
                                "title": f"Query time: Codec {c}"
                            }
                        ]
                    ) for i, c in enumerate(codecs)
                ]
            ],
            direction="down",
            showactive=True,
            x=0.4, xanchor="left", y=1.15, yanchor="top"
        )
    ],
    title_text="Upload Throughput per Codec and Size (Dataset: Azure)",
    xaxis_title="Codec",
    yaxis_title="Throughput (Mb/s)",
    barmode='group', # Keep them grouped when "All Sizes" is selected
    template="plotly_white"
)

fig.show()

### Download throughput per codec & per size

In [178]:
download_throughput_azure = {
    'S': azure_S["download_throughput"],
    'M': azure_M["download_throughput"],
    'L': azure_L["download_throughput"]
}

fig = go.Figure()

# 1. Add all traces initially (we will control visibility via dropdowns)
for size in sizes:
    fig.add_trace(
        go.Bar(
            x=codecs,
            y=download_throughput_azure[size],
            name=f"Size {size}",
            text=[f"{v:.3f}" for v in download_throughput_azure[size]],
            textposition='auto'
        )
    )

# 2. Define the Dropdown Menus
fig.update_layout(
    updatemenus=[
        # Dropdown 1: Select File Size (S, M, L, or All)
        dict(
            buttons=[
                dict(label="All Sizes", method="update", args=[{"visible": [True, True, True]}, {"title": "Download Throughput: All Sizes (Dataset: Azure)"}]),
                dict(label="Size S", method="update", args=[{"visible": [True, False, False]}, {"title": "Download Throughput: Size S"}]),
                dict(label="Size M", method="update", args=[{"visible": [False, True, False]}, {"title": "Download Throughput: Size M"}]),
                dict(label="Size L", method="update", args=[{"visible": [False, False, True]}, {"title": "Download Throughput: Size L"}]),
            ],
            direction="down",
            showactive=True,
            x=0.1, xanchor="left", y=1.15, yanchor="top"
        ),
        # Dropdown 2: Codec Filter (Restricts view to specific codec)
        dict(
            buttons=[
                dict(
                    label="All Codecs", 
                    method="update", 
                    args=[
                        {}, # No changes to trace data
                        {"xaxis.range": [-0.5, len(codecs)-0.5], "title": "Download Throughput: All Codecs (Dataset: Azure)"}
                    ]
                ),
                *[
                    dict(
                        label=f"Codec: {c}",
                        method="update",
                        args=[
                            {}, # No changes to trace data
                            {
                                "xaxis.range": [i-0.5, i+0.5], 
                                "title": f"Query time: Codec {c}"
                            }
                        ]
                    ) for i, c in enumerate(codecs)
                ]
            ],
            direction="down",
            showactive=True,
            x=0.4, xanchor="left", y=1.15, yanchor="top"
        )
    ],
    title_text="Download Throughput per Codec and Size (Dataset: Azure)",
    xaxis_title="Codec",
    yaxis_title="Throughput (Mb/s)",
    barmode='group', # Keep them grouped when "All Sizes" is selected
    template="plotly_white"
)

fig.show()

### Query runtime per codec & per size

In [179]:
query_time_azure = {
    'S': azure_S["query_time_seconds"],
    'M': azure_M["query_time_seconds"],
    'L': azure_L["query_time_seconds"]
}

fig = go.Figure()

# 1. Add all traces initially (we will control visibility via dropdowns)
for size in sizes:
    fig.add_trace(
        go.Bar(
            x=codecs,
            y=query_time_azure[size],
            name=f"Size {size}",
            text=[f"{v:.3f}" for v in query_time_azure[size]],
            textposition='auto'
        )
    )

# 2. Define the Dropdown Menus
fig.update_layout(
    updatemenus=[
        # Dropdown 1: Select File Size (S, M, L, or All)
        dict(
            buttons=[
                dict(label="All Sizes", method="update", args=[{"visible": [True, True, True]}, {"title": "Query Time: All Sizes (Dataset: Azure)"}]),
                dict(label="Size S", method="update", args=[{"visible": [True, False, False]}, {"title": "Query Time: Size S"}]),
                dict(label="Size M", method="update", args=[{"visible": [False, True, False]}, {"title": "Query Time: Size M"}]),
                dict(label="Size L", method="update", args=[{"visible": [False, False, True]}, {"title": "Query Time: Size L"}]),
            ],
            direction="down",
            showactive=True,
            x=0.1, xanchor="left", y=1.15, yanchor="top"
        ),
        # Dropdown 2: Codec Filter (Restricts view to specific codec)
        dict(
            buttons=[
                dict(
                    label="All Codecs", 
                    method="update", 
                    args=[
                        {}, # No changes to trace data
                        {"xaxis.range": [-0.5, len(codecs)-0.5], "title": "Query time: All Codecs (Dataset: Azure)"}
                    ]
                ),
                *[
                    dict(
                        label=f"Codec: {c}",
                        method="update",
                        args=[
                            {}, # No changes to trace data
                            {
                                "xaxis.range": [i-0.5, i+0.5], 
                                "title": f"Query time: Codec {c}"
                            }
                        ]
                    ) for i, c in enumerate(codecs)
                ]
            ],
            direction="down",
            showactive=True,
            x=0.4, xanchor="left", y=1.15, yanchor="top"
        )
    ],
    title_text="Query time per Codec and Size (Dataset: Azure)",
    xaxis_title="Codec",
    yaxis_title="Time (seconds)",
    barmode='group', # Keep them grouped when "All Sizes" is selected
    template="plotly_white"
)

fig.show()

# Normal Plots

## MinIo

## Size = S

In [180]:
# Barplot of the stored Mb per codec for the files with size S

plt.figure(figsize=(8, 6))
barsSMin = plt.bar(minio_S["codec"], minio_S["stored_Mb"]) # we can add color='color'  at the end to change it

# Annotating the bars
for bar in barsSMin:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=12)

# Adding labels and title
plt.xlabel("Codec", fontsize=14)
plt.ylabel("stored Mb", fontsize=14)
plt.title("stored Mb by codec (S)", fontsize=16)
plt.show()

In [181]:
# Grouped barplot comparing the upload and download throughput per codec for the files with size S

x_S_minio = np.arange(len(minio_S["codec"]))
width = 0.35

fig, ax = plt.subplots()
upload_sec_S_minio = ax.bar(x_S_minio-width/2, minio_S["upload_throughput"], width, label="Upload")
download_sec_S_minio = ax.bar(x_S_minio+width/2, minio_S["download_throughput"], width, label="Download")

ax.set_xticks(x_S_minio)
ax.set_xticklabels(minio_S['codec'])
ax.set_ylabel("Throughput (MB/s)")
ax.set_title("Upload vs Download throughput per Codec (S)")
ax.legend()
ax.bar_label(upload_sec_S_minio,fmt="%.2f", padding=3)
ax.bar_label(download_sec_S_minio,fmt="%.2f", padding=3)

plt.show()

## Size = M

In [182]:
# Barplot of the stored Mb per codec for the files with size M

plt.figure(figsize=(8, 6))
barsMMin = plt.bar(minio_M["codec"], minio_M["stored_Mb"]) # we can add color='color'  at the end to change it

# Annotating the bars
for bar in barsMMin:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=12)

# Adding labels and title
plt.xlabel("Codec", fontsize=14)
plt.ylabel("Mb", fontsize=14)
plt.title("stored_Mb by codec (M)", fontsize=16)
plt.show()

In [183]:
# Grouped barplot comparing the upload and download throughput per codec for the files with size M

x_M_minio = np.arange(len(minio_M["codec"]))
width = 0.35

fig, ax = plt.subplots()
upload_sec_M_minio = ax.bar(x_M_minio-width/2, minio_M["upload_throughput"], width, label="Upload")
download_sec_M_minio = ax.bar(x_M_minio+width/2, minio_M["download_throughput"], width, label="Download")

ax.set_xticks(x_M_minio)
ax.set_xticklabels(minio_M['codec'])
ax.set_ylabel("Throughput")
ax.set_title("Upload vs Download throughput per Codec (M)")
ax.legend()
ax.bar_label(upload_sec_M_minio,fmt="%.3f", padding=3)
ax.bar_label(download_sec_M_minio,fmt="%.3f", padding=3)

plt.show()

## Size = L

In [184]:
# Barplot of the stored Mb per codec for the files with size L

plt.figure(figsize=(8, 6))
barsLMin = plt.bar(minio_L["codec"], minio_L["stored_Mb"]) # we can add color='color' at the end to change it

# Annotating the bars
for bar in barsLMin:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=12)

# Adding labels and title
plt.xlabel("Codec", fontsize=14)
plt.ylabel("Mb", fontsize=14)
plt.title("Stored Mb by codec (L)", fontsize=16)
plt.show()

In [185]:
# Grouped barplot comparing the upload and download throughput per codec for the files with size L

x_L_minio = np.arange(len(minio_L["codec"]))
width = 0.35

fig, ax = plt.subplots()
upload_sec_L_minio = ax.bar(x_L_minio-width/2, minio_L["upload_throughput"], width, label="Upload")
download_sec_L_minio = ax.bar(x_L_minio+width/2, minio_L["download_throughput"], width, label="Download")

ax.set_xticks(x_L_minio)
ax.set_xticklabels(minio_L['codec'])
ax.set_ylabel("Throughput")
ax.set_title("Upload vs Download throughput per Codec (L)")
ax.legend()
ax.bar_label(upload_sec_L_minio,fmt="%.3f", padding=3)
ax.bar_label(download_sec_L_minio,fmt="%.3f", padding=3)

plt.show()

## Upload & Download per codec & per size

In [186]:
# Grouped barplot comparing the upload throughput per codec and per file size

x_M_minio = np.arange(len(minio_M["codec"]))*2 # doesn't matter if it's results_M, S or L since the codecs are the same.
# we *2 to increase the spacing between each group of bars (a group of bars = the bars of a codec)
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
upload_sec_S_minio = ax.bar(x_M_minio-width, minio_S["upload_throughput"], width, label="Upload throughput (S)")
upload_sec_M_minio = ax.bar(x_M_minio, minio_M["upload_throughput"], width, label="Upload throughput (M)")
upload_sec_L_minio = ax.bar(x_M_minio+width, minio_L["upload_throughput"], width, label="Upload throughput (L)")

ax.set_xticks(x_M_minio)
ax.set_xticklabels(minio_M['codec'])
ax.set_ylabel("Throughput")
ax.set_title("Upload throughput per Codec & per size")
ax.legend()
ax.bar_label(upload_sec_S_minio,fmt="%.3f", padding=3)
ax.bar_label(upload_sec_M_minio,fmt="%.3f", padding=3)
ax.bar_label(upload_sec_L_minio,fmt="%.3f", padding=3)

plt.show()

In [187]:
# Grouped barplot comparing the download per codec and per file size

x_M_minio = np.arange(len(minio_M["codec"]))*2 # doesn't matter if it's results_M, S or L since the codecs are the same.
# we *2 to increase the spacing between each group of bars (a group of bars = the bars of a codec)
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
download_sec_S_minio = ax.bar(x_M_minio-width, minio_S["download_throughput"], width, label="Download throughput (S)")
download_sec_M = ax.bar(x_M_minio, minio_M["download_throughput"], width, label="Download throughput (M)")
download_sec_L = ax.bar(x_M_minio+width, minio_L["download_throughput"], width, label="Download throughput (L)")

ax.set_xticks(x_M_minio)
ax.set_xticklabels(minio_M['codec'])
ax.set_ylabel("Throughput")
ax.set_title("Download throughput per Codec & per size")
ax.legend()
ax.bar_label(download_sec_S_minio,fmt="%.3f", padding=3)
ax.bar_label(download_sec_M_minio,fmt="%.3f", padding=3)
ax.bar_label(download_sec_L_minio,fmt="%.3f", padding=3)

plt.show()

## Query runtime per codec & per size

In [188]:
# Grouped barplot comparing the query runtime per codec and per file size

x_M_minio = np.arange(len(minio_M["codec"]))*2 # doesn't matter if it's results_M, S or L since the codecs are the same.
# we *2 to increase the spacing between each group of bars (a group of bars = the bars of a codec)
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
query_time_S_minio = ax.bar(x_M_minio-width, minio_S["query_time_seconds"], width, label="Query time (S)")
query_time_M_minio = ax.bar(x_M_minio, minio_M["query_time_seconds"], width, label="Query time (M)")
query_time_L_minio = ax.bar(x_M_minio+width, minio_L["query_time_seconds"], width, label="Query time (L)")

ax.set_xticks(x_M_minio)
ax.set_xticklabels(minio_M['codec'])
ax.set_ylabel("Time (seconds)")
ax.set_title("Query time per Codec (S, M & L)")
ax.legend()
ax.bar_label(query_time_S_minio,fmt="%.3f", padding=3)
ax.bar_label(query_time_M_minio,fmt="%.3f", padding=3)
ax.bar_label(query_time_L_minio,fmt="%.3f", padding=3)

plt.show()

## Azure

## Size = S

In [189]:
# Barplot of the stored Mb per codec for the files with size S

plt.figure(figsize=(8, 6))
barsSAz = plt.bar(azure_S["codec"], azure_S["stored_Mb"]) # we can add color='color'  at the end to change it

# Annotating the bars
for bar in barsSAz:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=12)

# Adding labels and title
plt.xlabel("Codec", fontsize=14)
plt.ylabel("stored Mb", fontsize=14)
plt.title("stored Mb by codec (S)", fontsize=16)
plt.show()

In [190]:
# Grouped barplot comparing the upload and download throughput per codec for the files with size S

x_S_az = np.arange(len(minio_S["codec"]))
width = 0.35

fig, ax = plt.subplots()
upload_sec_S_azure = ax.bar(x_S_az-width/2, azure_S["upload_throughput"], width, label="Upload")
download_sec_S_azure = ax.bar(x_S_az+width/2, azure_S["download_throughput"], width, label="Download")

ax.set_xticks(x_S_az)
ax.set_xticklabels(azure_S['codec'])
ax.set_ylabel("Throughput (MB/s)")
ax.set_title("Upload vs Download throughput per Codec (S)")
ax.legend()
ax.bar_label(upload_sec_S_azure,fmt="%.2f", padding=3)
ax.bar_label(download_sec_S_azure,fmt="%.2f", padding=3)

plt.show()

## Size = M

In [191]:
# Barplot of the stored Mb per codec for the files with size M

plt.figure(figsize=(8, 6))
barsMAz = plt.bar(azure_M["codec"], azure_M["stored_Mb"]) # we can add color='color'  at the end to change it

# Annotating the bars
for bar in barsMAz:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=12)

# Adding labels and title
plt.xlabel("Codec", fontsize=14)
plt.ylabel("Mb", fontsize=14)
plt.title("stored_Mb by codec (M)", fontsize=16)
plt.show()

In [192]:
# Grouped barplot comparing the upload and download throughput per codec for the files with size M

x_M_azure = np.arange(len(azure_M["codec"]))
width = 0.35

fig, ax = plt.subplots()
upload_sec_M_azure = ax.bar(x_M_azure-width/2, azure_M["upload_throughput"], width, label="Upload")
download_sec_M_azure = ax.bar(x_M_minio+width/2, azure_M["download_throughput"], width, label="Download")

ax.set_xticks(x_M_azure)
ax.set_xticklabels(azure_M['codec'])
ax.set_ylabel("Throughput")
ax.set_title("Upload vs Download throughput per Codec (M)")
ax.legend()
ax.bar_label(upload_sec_M_azure,fmt="%.3f", padding=3)
ax.bar_label(download_sec_M_azure,fmt="%.3f", padding=3)

plt.show()

## Size = L

In [193]:
# Barplot of the stored Mb per codec for the files with size L

plt.figure(figsize=(8, 6))
barsLAz = plt.bar(azure_L["codec"], azure_L["stored_Mb"]) # we can add color='color' at the end to change it

# Annotating the bars
for bar in barsLAz:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=12)

# Adding labels and title
plt.xlabel("Codec", fontsize=14)
plt.ylabel("Mb", fontsize=14)
plt.title("Stored Mb by codec (L)", fontsize=16)
plt.show()

In [194]:
# Grouped barplot comparing the upload and download throughput per codec for the files with size L

x_L_azure = np.arange(len(azure_L["codec"]))
width = 0.35

fig, ax = plt.subplots()
upload_sec_L_azure = ax.bar(x_L_azure-width/2, azure_L["upload_throughput"], width, label="Upload")
download_sec_L_azure = ax.bar(x_L_azure+width/2, azure_L["download_throughput"], width, label="Download")

ax.set_xticks(x_L_azure)
ax.set_xticklabels(azure_L['codec'])
ax.set_ylabel("Throughput")
ax.set_title("Upload vs Download throughput per Codec (L)")
ax.legend()
ax.bar_label(upload_sec_L_azure,fmt="%.3f", padding=3)
ax.bar_label(download_sec_L_azure,fmt="%.3f", padding=3)

plt.show()

## Upload & Download per codec & per size

In [195]:
# Grouped barplot comparing the upload throughput per codec and per file size

x_M_azure = np.arange(len(azure_M["codec"]))*2 # doesn't matter if it's results_M, S or L since the codecs are the same.
# we *2 to increase the spacing between each group of bars (a group of bars = the bars of a codec)
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
upload_sec_S_azure = ax.bar(x_M_azure-width, azure_S["upload_throughput"], width, label="Upload throughput (S)")
upload_sec_M_azure = ax.bar(x_M_azure, azure_M["upload_throughput"], width, label="Upload throughput (M)")
upload_sec_L_azure = ax.bar(x_M_azure+width, azure_L["upload_throughput"], width, label="Upload throughput (L)")

ax.set_xticks(x_M_azure)
ax.set_xticklabels(azure_M['codec'])
ax.set_ylabel("Throughput")
ax.set_title("Upload throughput per Codec & per size")
ax.legend()
ax.bar_label(upload_sec_S_azure,fmt="%.3f", padding=3)
ax.bar_label(upload_sec_M_azure,fmt="%.3f", padding=3)
ax.bar_label(upload_sec_L_azure,fmt="%.3f", padding=3)

plt.show()

In [196]:
# Grouped barplot comparing the download per codec and per file size

x_M_azure = np.arange(len(azure_M["codec"]))*2 # doesn't matter if it's results_M, S or L since the codecs are the same.
# we *2 to increase the spacing between each group of bars (a group of bars = the bars of a codec)
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
download_sec_S_azure = ax.bar(x_M_azure-width, azure_S["download_throughput"], width, label="Download throughput (S)")
download_sec_M_azure = ax.bar(x_M_azure, azure_M["download_throughput"], width, label="Download throughput (M)")
download_sec_L_azure = ax.bar(x_M_azure+width, azure_L["download_throughput"], width, label="Download throughput (L)")

ax.set_xticks(x_M_azure)
ax.set_xticklabels(azure_M['codec'])
ax.set_ylabel("Throughput")
ax.set_title("Download throughput per Codec & per size")
ax.legend()
ax.bar_label(download_sec_S_azure,fmt="%.3f", padding=3)
ax.bar_label(download_sec_M_azure,fmt="%.3f", padding=3)
ax.bar_label(download_sec_L_azure,fmt="%.3f", padding=3)

plt.show()

## Query runtime per codec & per size

In [197]:
# Grouped barplot comparing the query runtime per codec and per file size

x_M_azure = np.arange(len(azure_M["codec"]))*2 # doesn't matter if it's results_M, S or L since the codecs are the same.
# we *2 to increase the spacing between each group of bars (a group of bars = the bars of a codec)
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
query_time_S_azure = ax.bar(x_M_azure-width, azure_S["query_time_seconds"], width, label="Query time (S)")
query_time_M_azure = ax.bar(x_M_azure, azure_M["query_time_seconds"], width, label="Query time (M)")
query_time_L_azure = ax.bar(x_M_azure+width, azure_L["query_time_seconds"], width, label="Query time (L)")

ax.set_xticks(x_M_azure)
ax.set_xticklabels(azure_M['codec'])
ax.set_ylabel("Time (seconds)")
ax.set_title("Query time per Codec (S, M & L)")
ax.legend()
ax.bar_label(query_time_S_azure,fmt="%.3f", padding=3)
ax.bar_label(query_time_M_azure,fmt="%.3f", padding=3)
ax.bar_label(query_time_L_azure,fmt="%.3f", padding=3)

plt.show()